In [ ]:
"""
CS781 Phase 3 - Rhetorical Role Prediction (RR) with Model 2 + LoRA / AdaLoRA
=============================================================================

Base model : Model 2 (decoder-only, RoPE + GQA + SwiGLU, weight-tied, clean byte-level BPE 32k)
Method     : PEFT_METHOD = "rslora" (default), "adalora" or "lora", on every attention / FFN projection (base weights frozen)
             LoRA    : every matrix gets the same rank r, update scaled by alpha / r
             rsLoRA  : LoRA with the update scaled by alpha / sqrt(r) instead (rank-stabilized LoRA, Kalajdzievski, 2023). alpha / r makes
                       the update shrink as r grows, so large ranks learn no better than small ones; alpha / sqrt(r) keeps learning
                       stable at every rank, so extra rank is actually used
             AdaLoRA : dW = P diag(lam) Q (SVD form) with a global rank budget that shrinks during training; the least
                       important singular triplets are pruned, so important matrices keep a high rank (Zhang et al., 2023)
             + a small sentence-classification head (13 roles) trained from scratch
Task       : one label per sentence, scored with Macro-F1 on CL + IT test documents

How it works
------------
Model 2 is causal, so a sentence's representation can only use text to its LEFT.
We therefore feed the model a stretch of consecutive sentences (left context + a block of
"target" sentences), take the hidden state of the LAST token of every target sentence, and
classify it.  Every sentence is a target exactly once at inference time, and always has as
much left context as fits in the 1024-token window.

Meant for a Kaggle notebook (paste into one cell, or run as a script).  Attach to the notebook:
    * the dataset with Model 2's checkpoint  (best.pt + best_meta.json from the pretraining run)
    * the dataset with the BPE tokenizer     (tokenizer.json + meta.json from bpe-final.ipynb)
The Phase 3 data is downloaded from the HF dataset repo (Kaggle secret named HF) unless found locally.

Imbalance handling (v2): per-domain capped inverse-sqrt class weights, oversampling of windows that contain rare labels,
domain-aware handling of labels unseen in a domain, and a 2-parameter post-hoc calibration (tau, unseen-label mask) picked on dev.

Outputs (WORK_DIR):
    data_report.json            everything learned from the data (labels, counts, token stats, class weights)
    <method>_ckpt/best_adapter.pt   adapter + head weights of the best dev Macro-F1 (small, a few MB)
    <method>_ckpt/train_log.csv     per-epoch log
    rank_allocation.json            (AdaLoRA) final rank of every layer / projection
    prediction.json, submission.zip
"""
import os
import csv
import json
import math
import time
import zipfile
import random
from pathlib import Path
from dataclasses import dataclass
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================================
# CONFIG
# ============================================================================
SEED = 1234

IS_KAGGLE = os.path.isdir("/kaggle/working")
PEFT_METHOD = "rslora"       # "rslora" | "adalora" | "lora"   (same data pipeline, head, loss and post-processing for all -> fair comparison)
assert PEFT_METHOD in ("lora", "rslora", "adalora"), PEFT_METHOD
WORK_DIR = f"/kaggle/working/phase3_rr_{PEFT_METHOD}" if IS_KAGGLE else f"./phase3_rr_{PEFT_METHOD}"
CKPT_DIR = os.path.join(WORK_DIR, f"{PEFT_METHOD}_ckpt")
INPUT_ROOT = "/kaggle/input"

# ---- data (same repo / file names as phase3.ipynb) ----
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev": "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev": "Phase_3_RR/Train/IT_dev.json",
    "label_vocab": "Phase_3_RR/Train/label_vocab.json",
    "CL_test": "Phase_3_RR/Test/CL_test.json",
    "IT_test": "Phase_3_RR/Test/IT_test.json",
}
LOCAL_PHASE3_DIR = os.environ.get("PHASE3_DIR", "")      # optional: root that contains Phase_3_RR/

# ---- base model (Model 2) ----
MODEL2_TAG = "decoder_only_rope_gqa_swiglu_clean_bpe32k_v1"   # tag written by the pretraining notebook
MODEL2_CKPT_PATH = ""        # set to an explicit best.pt / latest.pt to skip the automatic search
TOKENIZER_JSON = ""          # set to an explicit tokenizer.json to skip the automatic search
EXPECTED_VOCAB = 32_000

# ---- windowing (tokens) ----
MAX_LEN = 1024               # Model 2 context length
MAX_SENT_TOK = 160           # a sentence is cut to its first 160 tokens (p99 is well below this)
SENT_PREFIX = " "            # prepended to every sentence before BPE, so words look like "mid-text" words
TARGET_TOK = 320             # tokens of target sentences per window at inference
TRAIN_TARGET_RANGE = (160, 480)   # random target block size per window while training (augmentation)

# ---- LoRA ----
LORA_R = 16
LORA_ALPHA = 32              # scaling = alpha / r
LORA_DROPOUT = 0.05
LORA_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "w_gate", "w_up", "w_down")
TRAIN_FINAL_NORM = True      # also tune the 640 RMSNorm weights before the head

# ---- rsLoRA (used when PEFT_METHOD == "rslora"; LORA_TARGETS, LORA_DROPOUT and TRAIN_FINAL_NORM are shared) ----
# Same module as LoRA (A kaiming-init, B zero-init); the ONLY difference is scaling = alpha / sqrt(r)  instead of  alpha / r.
RSLORA_R = 64                # the point of rsLoRA is to be able to use a larger rank than plain LoRA's 16 (try 32 / 128 / 256 too)
RSLORA_ALPHA = (LORA_ALPHA / LORA_R) * math.sqrt(LORA_R)    # = 8.0. Calibrated so that scaling(r=LORA_R) equals LoRA's 2.0:
                             # RSLORA_R = 16 then reproduces plain LoRA exactly, and changing RSLORA_R changes only the rank, not the tuning.
                             # (scaling = RSLORA_ALPHA / sqrt(RSLORA_R): r=16 -> 2.0, r=64 -> 1.0, r=256 -> 0.5; plain LoRA with alpha=32 would give 2.0, 0.5, 0.125)

# ---- AdaLoRA (used when PEFT_METHOD == "adalora"; LORA_TARGETS, LORA_DROPOUT and TRAIN_FINAL_NORM are shared) ----
ADALORA_INIT_R = 24          # initial rank of every adapted matrix  -> initial budget b(0) = n_matrices * INIT_R
ADALORA_TARGET_R = 16        # AVERAGE final rank per matrix         -> final budget  b(T) = n_matrices * TARGET_R
                             # (16 = the same total rank as LORA_R=16, so the comparison is 'same budget, smarter allocation')
ADALORA_ALPHA = 48           # scaling = alpha / INIT_R = 2.0 (same as LoRA's 32 / 16)
ADALORA_TINIT_FRAC = 0.10    # no pruning during the first 10% of optimiser steps (importance scores warm up)
ADALORA_TFINAL_FRAC = 0.35   # budget frozen for the last 35% of steps (survivors are fine-tuned with a fixed budget)
ADALORA_DELTA_T = 10         # re-allocate the budget every 10 optimiser steps
ADALORA_BETA = 0.85          # EMA factor of sensitivity and uncertainty
ADALORA_ORTH_REG = 0.1       # weight of the orthogonality penalty on P (lora_B) and Q (lora_A)
LR_ADALORA = 5e-4            # lam starts at 0, so AdaLoRA warms up slower than LoRA -> slightly higher LR; tune first if results are off
ADALORA_SELECT_AFTER_FREEZE = True   # best-dev-epoch selection only among epochs with the FINAL budget (else the saved model may exceed it)

# ---- head ----
POOLING = "last"             # "last" = hidden of last token | "last_mean" = concat(last, mean over sentence tokens)
USE_POS_FEATURE = True       # relative position of the sentence in its document (bucketed)
N_POS_BUCKETS = 20
USE_DOMAIN_FEATURE = True    # CL / IT embedding
HEAD_HIDDEN = 256
HEAD_DROPOUT = 0.2

# ---- optimisation ----
FT_DROPOUT = 0.1             # residual dropout of Model 2 while fine-tuning (pretraining used 0.2)
EPOCHS = 12
MICRO_BS = 4
ACCUM = 2
LR_LORA = 3e-4
LR_RSLORA = LR_LORA          # rsLoRA starts from the plain-LoRA LR (its alpha is calibrated to LoRA's scaling at r=16); tune first if results are off
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
# ---- class imbalance handling ----
DOMAINS = ["CL", "IT"]
CLASS_WEIGHT_POWER = 0.5     # weight_c = (N_d / (K_d * n_dc)) ** power : 0 = unweighted, 0.5 = inverse-sqrt, 1 = fully balanced
CLASS_WEIGHT_CLIP = (0.25, 8.0)   # cap: a label with 3 examples must not get a 100x gradient
PER_DOMAIN_WEIGHTS = True    # weights from each domain's OWN label counts (a label rare in IT but common in CL is up-weighted in IT only)
OVERSAMPLE_RARE = True       # repeat training windows that contain a rare label (rare = < RARE_FRAC of its domain's train sentences)
RARE_FRAC = 0.01
RARE_REPEAT = 3              # a window with a rare label is seen this many times per epoch (fresh dropout each time)
POSTHOC_CALIBRATION = True   # on dev: pick logit-adjustment strength tau and whether to mask labels unseen in a domain
TAU_GRID = (0.0, 0.25, 0.5, 0.75, 1.0)
LABEL_SMOOTHING = 0.0
PATIENCE = 4                 # stop when dev Macro-F1 has not improved for this many epochs
RUN_PREDICTION = True

AMP = {"enabled": False, "dtype": torch.float16}


def amp_ctx():
    return torch.autocast(device_type="cuda", dtype=AMP["dtype"], enabled=AMP["enabled"])


def log(*a):
    print(*a, flush=True)


def set_seed(s=SEED):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


# ============================================================================
# 1. DATA LOADING  (same layout handling as phase3.ipynb, token via Kaggle secret - never hard-coded)
# ============================================================================
def get_hf_token():
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or None


def resolve_phase3_files():
    paths = {}
    roots = []
    if LOCAL_PHASE3_DIR and Path(LOCAL_PHASE3_DIR).is_dir():
        roots.append(Path(LOCAL_PHASE3_DIR))
    roots += [Path(INPUT_ROOT), Path(WORK_DIR), Path.cwd()]
    wanted = {Path(p).name: k for k, p in PHASE3_FILES.items()}
    for root in roots:
        if not root.exists():
            continue
        for fp in root.rglob("*.json"):
            if fp.name in wanted:
                paths.setdefault(wanted[fp.name], str(fp))
    missing = [k for k in PHASE3_FILES if k not in paths]
    if missing:
        from huggingface_hub import hf_hub_download
        token = get_hf_token()
        for k in missing:
            log("downloading", PHASE3_FILES[k])
            paths[k] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=PHASE3_FILES[k],
                                       repo_type="dataset", token=token)
    return paths


def read_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)



def normalize_documents(raw, split_name):
    """Normalize Phase 3 RR JSON into:
       [{"id": ..., "text": [...], "labels": [...] or None}]
    """
    docs = []

    def is_sentence_list(value):
        return (
            isinstance(value, list)
            and all(isinstance(s, str) for s in value)
        )

    def visit(obj, inherited_id=None):
        if isinstance(obj, list):
            # A list of strings may be a document's sentence list.
            if is_sentence_list(obj):
                docs.append({
                    "id": str(inherited_id or f"{split_name}_{len(docs)}"),
                    "text": obj,
                    "labels": None,
                })
                return

            for item in obj:
                visit(item, inherited_id)
            return

        if not isinstance(obj, dict):
            # Ignore metadata strings and other scalar values.
            return

        text = obj.get("text", obj.get("sentences"))
        labels = obj.get("labels", obj.get("label"))

        # Some files store sentences under a document field.
        if text is None:
            for key in ("data", "documents", "docs", "items"):
                if key in obj:
                    visit(obj[key], inherited_id or obj.get("id"))
                    return

            # Wrapper dictionaries may contain documents under arbitrary keys.
            for key, value in obj.items():
                if isinstance(value, (dict, list)):
                    visit(value, str(key))
            return

        if isinstance(text, str):
            text = [text]

        if not isinstance(text, list) or not all(
            isinstance(s, str) for s in text
        ):
            raise TypeError(
                f"{split_name}: invalid text format for document "
                f"{obj.get('id', inherited_id)}"
            )

        # Normalize labels only when present and valid.
        if isinstance(labels, list):
            labels = [str(x) for x in labels]
        else:
            labels = None

        docs.append({
            "id": str(obj.get("id", inherited_id or
                              f"{split_name}_{len(docs)}")),
            "text": text,
            "labels": labels,
        })

    visit(raw)

    if not docs:
        raise ValueError(f"{split_name}: no documents found")

    # Check IDs and label alignment.
    ids = [d["id"] for d in docs]
    if len(ids) != len(set(ids)):
        raise ValueError(f"{split_name}: duplicate document IDs detected")

    for d in docs:
        if d["labels"] is not None and len(d["labels"]) != len(d["text"]):
            raise ValueError(
                f"{split_name}: label/text length mismatch in {d['id']}: "
                f"{len(d['labels'])} labels vs {len(d['text'])} sentences"
            )

    return docs



def load_labels(raw):
    if isinstance(raw, dict) and "labels" in raw:
        raw = raw["labels"]
    if isinstance(raw, dict):
        raw = [k for k, _ in sorted(raw.items(), key=lambda kv: kv[1])]
    labels = [str(x) for x in raw]
    assert len(labels) == len(set(labels))
    return labels


def load_phase3():
    paths = resolve_phase3_files()
    raw = {k: read_json(p) for k, p in paths.items()}
    labels = load_labels(raw["label_vocab"])
    data = {k: normalize_documents(raw[k], k) for k in raw if k != "label_vocab"}
    l2i = {l: i for i, l in enumerate(labels)}
    for name, docs in data.items():
        labelled = name.endswith("_train") or name.endswith("_dev")
        for d in docs:
            d["domain"] = name.split("_")[0]
            if labelled:
                assert d["labels"] is not None and len(d["labels"]) == len(d["text"]), (name, d["id"])
                bad = [x for x in d["labels"] if x not in l2i]
                assert not bad, f"{name}/{d['id']}: unknown labels {set(bad)}"
    return data, labels, l2i


# ============================================================================
# 2. MODEL 2 ARCHITECTURE  (parameter names identical to the pretraining notebook so best.pt loads strictly;
#    the KV-cache generation path is dropped because classification does not need it)
# ============================================================================
@dataclass
class ModelConfig:
    vocab_size: int = 32_000
    d_model: int = 640
    n_layers: int = 8
    n_heads: int = 10
    n_kv_heads: int = 2
    ffn_hidden: int = 2048
    dropout: float = 0.2
    max_len: int = 1024 + 64
    pad_id: int = 0
    unk_id: int = 1
    eos_id: int = 2


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight).type_as(x)


def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    out = torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)
    return out.type_as(x)


def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h = cfg.n_heads
        self.kv = cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.norm2 = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def hidden(self, idx):
        """(B, T) token ids -> (B, T, d_model) final-norm hidden states (what the tied LM head would read)"""
        T = idx.size(1)
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        x = self.drop(self.embed(idx))
        for blk in self.blocks:
            x = blk(x, cos, sin)
        return self.final_norm(x)


# ============================================================================
# 3. LoRA / rsLoRA
# ============================================================================
class LoRALinear(nn.Module):
    """y = W x + scaling * B(A(dropout(x)));  W frozen, A: (r, in) kaiming-init, B: (out, r) zero-init
         scaling = alpha / r          standard LoRA
         scaling = alpha / sqrt(r)    rank_stabilized=True -> rsLoRA (Kalajdzievski, 2023)
    -> at step 0 the adapted model is EXACTLY the base model."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float, rank_stabilized: bool = False):
        super().__init__()
        assert base.bias is None, "Model 2 uses bias-free projections"
        self.base = base
        self.r = r
        self.rank_stabilized = rank_stabilized
        self.scaling = alpha / (math.sqrt(r) if rank_stabilized else r)
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    @property
    def weight(self):                      # lets code that peeks at .weight keep working
        return self.base.weight

    def forward(self, x):
        out = self.base(x)
        if self.merged:
            return out
        lora = F.linear(F.linear(self.drop(x), self.lora_A.to(x.dtype)), self.lora_B.to(x.dtype))
        return out + lora * self.scaling

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += (self.lora_B @ self.lora_A) * self.scaling
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight -= (self.lora_B @ self.lora_A) * self.scaling
            self.merged = False


def inject_lora(lm: DecoderLM, targets=LORA_TARGETS, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT,
                rank_stabilized=False):
    """freeze everything in the LM, then wrap every Linear whose attribute name is in `targets`
    (rank_stabilized=True -> rsLoRA scaling alpha / sqrt(r))"""
    for p in lm.parameters():
        p.requires_grad = False
    n = 0
    for blk in lm.blocks:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if name in targets and isinstance(child, nn.Linear):
                    setattr(parent, name, LoRALinear(child, r, alpha, dropout, rank_stabilized))
                    n += 1
    assert n > 0, "no Linear layer matched LORA_TARGETS"
    if TRAIN_FINAL_NORM:
        lm.final_norm.weight.requires_grad = True
    return n


def lora_modules(model):
    return [m for m in model.modules() if isinstance(m, (LoRALinear, AdaLoRALinear))]


def merge_lora(model):
    for m in lora_modules(model):
        m.merge()


def unmerge_lora(model):
    for m in lora_modules(model):
        m.unmerge()


# ============================================================================
# 3b. AdaLoRA   (Zhang et al., ICLR 2023 - "Adaptive Budget Allocation for Parameter-Efficient Fine-Tuning")
# ============================================================================
# LoRA gives every adapted matrix the same rank r.  AdaLoRA writes the update in SVD form
#       dW = P diag(lam) Q        P = lora_B (out, r)     lam = lora_E (r,)     Q = lora_A (r, in)
# and shrinks a GLOBAL rank budget during training.  Every (P[:, i], lam_i, Q[i, :]) is one "triplet"; the triplets with
# the lowest importance are switched off (lam_i = 0), so matrices that matter keep many triplets and the others few or none.
#   sensitivity of a weight   I(w) = |w * dL/dw|      smoothed:  I_bar = b*I_bar + (1-b)*I,   U = b*U + (1-b)*|I - I_bar|
#   score of a weight         s(w) = I_bar(w) * U(w)
#   score of triplet i        S_i  = s(lam_i) + mean_j s(P_ji) + mean_j s(Q_ij)
#   budget schedule b(t)      b(0) until t_init, cubic decay to b(T), constant b(T) for the last t_final steps
#   orthogonality penalty     ||P^T P - I||_F + ||Q Q^T - I||_F     (keeps P, Q close to real singular vectors)
# After the last pruning step the surviving triplets are fixed (rank_mask) and fine-tuned for t_final more steps.
class AdaLoRALinear(nn.Module):
    """y = W x + (alpha / r_init) * P diag(lam * rank_mask) Q dropout(x);  W frozen, lam zero-init -> EXACTLY the base model at step 0.
    Parameter names start with 'lora_' on purpose: trainable_state / freezing checks / optimiser grouping treat it like LoRA."""

    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        assert base.bias is None, "Model 2 uses bias-free projections"
        self.base = base
        self.r = r
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))        # Q
        self.lora_B = nn.Parameter(torch.empty(base.out_features, r))       # P
        self.lora_E = nn.Parameter(torch.zeros(r))                          # lam (singular values)
        nn.init.normal_(self.lora_A, std=0.02)
        nn.init.normal_(self.lora_B, std=0.02)
        self.register_buffer("rank_mask", torch.ones(r))                    # 0 = triplet removed for good (set by the final budget freeze)
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    @property
    def weight(self):                      # lets code that peeks at .weight keep working
        return self.base.weight

    @property
    def lam(self):
        return self.lora_E * self.rank_mask

    def forward(self, x):
        out = self.base(x)
        if self.merged:
            return out
        z = F.linear(self.drop(x), self.lora_A.to(x.dtype)) * self.lam.to(x.dtype)     # (..., r)
        return out + F.linear(z, self.lora_B.to(x.dtype)) * self.scaling

    def delta_weight(self):
        return ((self.lora_B * self.lam[None, :]) @ self.lora_A) * self.scaling

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += self.delta_weight()
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight -= self.delta_weight()
            self.merged = False

    def active_rank(self):
        return int((self.lam != 0).sum().item())


def inject_adalora(lm: DecoderLM, targets=LORA_TARGETS, r=ADALORA_INIT_R, alpha=ADALORA_ALPHA, dropout=LORA_DROPOUT):
    """freeze everything in the LM, then wrap every Linear whose attribute name is in `targets` (same sites as inject_lora)"""
    for p in lm.parameters():
        p.requires_grad = False
    n = 0
    for blk in lm.blocks:
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if name in targets and isinstance(child, nn.Linear):
                    setattr(parent, name, AdaLoRALinear(child, r, alpha, dropout))
                    n += 1
    assert n > 0, "no Linear layer matched LORA_TARGETS"
    if TRAIN_FINAL_NORM:
        lm.final_norm.weight.requires_grad = True
    return n


def adalora_budget(step, total_steps, init_budget, target_budget, t_init, t_final):
    """cubic budget schedule of the paper -> (rank budget at this step, True once pruning is active)"""
    if step < t_init:
        return init_budget, False
    if step >= total_steps - t_final:
        return target_budget, True
    frac = 1.0 - (step - t_init) / max(1, total_steps - t_final - t_init)
    return int(target_budget + (init_budget - target_budget) * frac ** 3), True


class AdaLoRAAllocator:
    """importance scores + global budget pruning.  Usage per optimiser step (after backward, before zero_grad):
           allocator.update_and_allocate(step)"""

    def __init__(self, lm, total_steps):
        self.mods = [m for m in lm.modules() if isinstance(m, AdaLoRALinear)]
        assert self.mods, "no AdaLoRALinear found - call inject_adalora first"
        assert ADALORA_TARGET_R <= ADALORA_INIT_R, "ADALORA_TARGET_R must be <= ADALORA_INIT_R"
        self.n = len(self.mods)
        self.init_budget = sum(m.r for m in self.mods)
        self.target_budget = self.n * ADALORA_TARGET_R
        self.total_steps = int(total_steps)
        self.t_init = int(ADALORA_TINIT_FRAC * self.total_steps)
        self.t_final = int(ADALORA_TFINAL_FRAC * self.total_steps)
        assert self.t_init + self.t_final < self.total_steps, "ADALORA_TINIT_FRAC + ADALORA_TFINAL_FRAC must be < 1"
        self.ema = {}              # (module index, "E" | "A" | "B") -> [EMA of sensitivity, EMA of uncertainty]
        self.frozen = False

    # ---- scores ----
    @torch.no_grad()
    def update_ipt(self):
        b = ADALORA_BETA
        for i, m in enumerate(self.mods):
            for key, p in (("E", m.lora_E), ("A", m.lora_A), ("B", m.lora_B)):
                if p.grad is None:
                    continue
                ipt = (p * p.grad).abs()                                    # |w * dL/dw|
                if (i, key) not in self.ema:
                    self.ema[(i, key)] = [torch.zeros_like(p), torch.zeros_like(p)]
                s, u = self.ema[(i, key)]
                s.mul_(b).add_(ipt, alpha=1 - b)
                u.mul_(b).add_((ipt - s).abs(), alpha=1 - b)

    def triplet_scores(self):
        out = []
        for i, m in enumerate(self.mods):
            if any((i, k) not in self.ema for k in "EAB"):
                out.append(torch.zeros(m.r, device=m.lora_E.device))
                continue
            sE = self.ema[(i, "E")][0] * self.ema[(i, "E")][1]                          # (r,)
            sA = (self.ema[(i, "A")][0] * self.ema[(i, "A")][1]).mean(dim=1)            # (r,)  mean over the input dim
            sB = (self.ema[(i, "B")][0] * self.ema[(i, "B")][1]).mean(dim=0)            # (r,)  mean over the output dim
            out.append(sE + sA + sB)
        return out

    # ---- pruning ----
    @torch.no_grad()
    def mask_to_budget(self, budget):
        """keep the `budget` highest-scoring triplets over ALL matrices, zero lam of the rest (they may regrow until the freeze)"""
        scores = self.triplet_scores()
        flat = torch.cat(scores)
        budget = int(max(0, min(budget, flat.numel())))
        keep = torch.zeros(flat.numel(), dtype=torch.bool, device=flat.device)
        if budget > 0:
            keep[torch.topk(flat, budget).indices] = True
        keeps, ofs = [], 0
        for m in self.mods:
            k = keep[ofs:ofs + m.r]
            ofs += m.r
            m.lora_E.masked_fill_(~k, 0.0)
            keeps.append(k)
        return keeps

    @torch.no_grad()
    def freeze(self):
        """final pruning to the target budget; afterwards the surviving set is permanent (rank_mask) and only fine-tuned"""
        keeps = self.mask_to_budget(self.target_budget)
        for m, k in zip(self.mods, keeps):
            m.rank_mask.copy_(k.to(m.rank_mask.dtype))
        self.frozen = True
        self.ema.clear()

    def update_and_allocate(self, step):
        if self.frozen:
            return
        if step >= self.total_steps - self.t_final:
            self.freeze()
            return
        self.update_ipt()
        budget, pruning = adalora_budget(step, self.total_steps, self.init_budget, self.target_budget,
                                         self.t_init, self.t_final)
        if pruning and step % ADALORA_DELTA_T == 0:
            self.mask_to_budget(budget)

    # ---- regulariser ----
    def orth_reg(self):
        """mean over all P and Q matrices of ||P^T P - I||_F, ||Q Q^T - I||_F (official AdaLoRA form; eps keeps the gradient finite)"""
        total, count = 0.0, 0
        for m in self.mods:
            A, B = m.lora_A.float(), m.lora_B.float()
            eye = torch.eye(m.r, device=A.device)
            for gram in (A @ A.t(), B.t() @ B):
                total = total + ((gram - eye).pow(2).sum() + 1e-12).sqrt()
                count += 1
        return total / count

    # ---- logging ----
    def describe(self):
        return (f"AdaLoRA: {self.n} matrices | total rank budget {self.init_budget} (r_init={ADALORA_INIT_R}) -> {self.target_budget} "
                f"(avg r={ADALORA_TARGET_R}) | no pruning before step {self.t_init}, cubic decay until step "
                f"{self.total_steps - self.t_final}, budget frozen afterwards | re-allocation every {ADALORA_DELTA_T} steps")

    def status(self, step):
        budget, _ = adalora_budget(step, self.total_steps, self.init_budget, self.target_budget, self.t_init, self.t_final)
        active = sum(m.active_rank() for m in self.mods)
        return (f"AdaLoRA: scheduled budget {budget} | active triplets {active}/{self.init_budget} | "
                f"budget frozen: {self.frozen}")


def peft_meta():
    if PEFT_METHOD == "rslora":
        return dict(method="rslora", r=RSLORA_R, alpha=RSLORA_ALPHA, rank_stabilized=True,
                    scaling=RSLORA_ALPHA / math.sqrt(RSLORA_R), targets=list(LORA_TARGETS))
    if PEFT_METHOD == "adalora":
        return dict(method="adalora", init_r=ADALORA_INIT_R, target_r=ADALORA_TARGET_R, alpha=ADALORA_ALPHA,
                    targets=list(LORA_TARGETS))
    return dict(method="lora", r=LORA_R, alpha=LORA_ALPHA, targets=list(LORA_TARGETS))


def _perturb(m):
    """make an adapter non-trivial (used by the self-checks only)"""
    if isinstance(m, AdaLoRALinear):
        m.lora_E.add_(torch.randn_like(m.lora_E) * 0.3)
    else:
        m.lora_B.add_(torch.randn_like(m.lora_B) * 0.01)


def _unperturb(m):
    (m.lora_E if isinstance(m, AdaLoRALinear) else m.lora_B).zero_()


def run_adalora_checks(model, device):
    """budget schedule, exact-budget pruning, permanence of the final freeze, finite orthogonality loss - then restore the pristine state"""
    lm = model.lm
    model.eval()
    mods = [m for m in lm.modules() if isinstance(m, AdaLoRALinear)]
    init_b, tgt_b = sum(m.r for m in mods), len(mods) * ADALORA_TARGET_R

    tot, ti, tf = 1000, 100, 350                                           # 1. schedule
    seq = [adalora_budget(s, tot, init_b, tgt_b, ti, tf)[0] for s in range(tot + 50)]
    assert seq[0] == init_b and seq[ti - 1] == init_b and seq[tot - tf] == tgt_b and seq[-1] == tgt_b
    assert all(a >= b for a, b in zip(seq, seq[1:])), "budget schedule is not monotone"

    ids = torch.randint(3, lm.cfg.vocab_size, (1, 64), device=device)
    saved_amp, AMP["enabled"] = AMP["enabled"], False
    try:
        alloc = AdaLoRAAllocator(lm, total_steps=tot)

        def fresh_grads():                                                  # random non-zero lam, then one backward -> grads for E, A and B
            with torch.no_grad():
                for m in mods:
                    m.lora_E.copy_(torch.randn_like(m.lora_E))
            with torch.enable_grad():
                h = lm.hidden(ids)
                (h.float().pow(2).mean() + ADALORA_ORTH_REG * alloc.orth_reg()).backward()

        fresh_grads()                                                       # 2. exactly `budget` triplets survive
        assert torch.isfinite(alloc.orth_reg()).item(), "orthogonality loss is not finite"
        assert all(m.lora_A.grad is not None and torch.isfinite(m.lora_A.grad).all() for m in mods), "bad gradients"
        alloc.update_ipt()
        budget = tgt_b - 7
        alloc.mask_to_budget(budget)
        active = sum(int((m.lora_E != 0).sum()) for m in mods)
        assert active == budget, f"pruning kept {active} triplets, expected {budget}"

        fresh_grads()                                                       # 3. final freeze is permanent
        alloc.ema.clear()
        alloc.update_ipt()
        alloc.freeze()
        assert alloc.frozen and int(sum(m.rank_mask.sum().item() for m in mods)) == tgt_b
        with torch.no_grad():
            before = lm.hidden(ids)
            for m in mods:                                                  # "regrow" pruned lam: must have no effect any more
                m.lora_E.add_(torch.randn_like(m.lora_E) * (m.rank_mask == 0))
            after = lm.hidden(ids)
        assert (before - after).abs().max() < 1e-6, "pruned triplets still influence the output after the freeze"
    finally:
        AMP["enabled"] = saved_amp
        for m in mods:                                                      # restore the pristine injected state
            m.lora_E.data.zero_()
            m.rank_mask.fill_(1.0)
            for p in (m.lora_A, m.lora_B, m.lora_E):
                p.grad = None
        lm.final_norm.weight.grad = None
    log(f"AdaLoRA self-checks passed: schedule {init_b} -> {tgt_b} is monotone, pruning keeps exactly the budget, "
        f"the freeze is permanent, orthogonality loss/gradients are finite")


def report_rank_allocation(model, path):
    """which layers / projections ended up with how many singular triplets (the thing AdaLoRA learns) + parameter comparison"""
    names = list(LORA_TARGETS)
    table, tot_active, tot_init, p_ada, p_lora = {}, 0, 0, 0, 0
    for li, blk in enumerate(model.lm.blocks):
        for parent in (blk.attn, blk.ffn):
            for name, child in parent.named_children():
                if isinstance(child, AdaLoRALinear):
                    a = child.active_rank()
                    table.setdefault(li, {})[name] = a
                    tot_active += a
                    tot_init += child.r
                    d_in, d_out = child.base.in_features, child.base.out_features
                    p_ada += a * (d_in + d_out + 1)
                    p_lora += LORA_R * (d_in + d_out)
    log("\nAdaLoRA final rank per matrix (active singular triplets):")
    log("  layer " + " ".join(f"{n:>7s}" for n in names) + "   total")
    for li in sorted(table):
        row = [table[li].get(n, 0) for n in names]
        log(f"  L{li:<4d} " + " ".join(f"{v:7d}" for v in row) + f"   {sum(row):5d}")
    mean_by_module = {n: float(np.mean([table[li].get(n, 0) for li in table])) for n in names}
    mean_by_layer = {int(li): float(np.mean([table[li].get(n, 0) for n in names])) for li in table}
    log("  mean rank by projection: " + ", ".join(f"{n}={v:.1f}" for n, v in mean_by_module.items()))
    log(f"  active triplets {tot_active}/{tot_init} | AdaLoRA adapter params {p_ada / 1e6:.2f}M "
        f"(uniform LoRA r={LORA_R} would be {p_lora / 1e6:.2f}M)")
    with open(path, "w") as f:
        json.dump({"rank_per_layer_and_projection": table, "mean_rank_by_projection": mean_by_module,
                   "mean_rank_by_layer": mean_by_layer, "active_triplets": tot_active, "initial_triplets": tot_init,
                   "adalora_adapter_params": p_ada, f"uniform_lora_r{LORA_R}_params": p_lora}, f, indent=2)



# ============================================================================
# 4. TOKENIZER + DATA -> TOKEN WINDOWS
# ============================================================================
class LegalByteBPE:
    def __init__(self, tokenizer_json, meta_path=None):
        from tokenizers import Tokenizer
        self.tokenizer = Tokenizer.from_file(tokenizer_json)
        self.vocab_size = self.tokenizer.get_vocab_size()
        vocab = self.tokenizer.get_vocab()
        self.pad_id = vocab["<|pad|>"]
        self.unk_id = vocab["<|unk|>"]
        self.eos_id = vocab["<|endoftext|>"]
        if meta_path and os.path.isfile(meta_path):
            meta = read_json(meta_path)
            assert meta["vocab_size"] == self.vocab_size

    def encode(self, text):
        # drop pad/eos in case the tokenizer's post-processor appends them: pooling reads the LAST token of a sentence
        return [i for i in self.tokenizer.encode(text).ids if i not in (self.pad_id, self.eos_id)]


def load_tokenizer():
    cands = []
    if TOKENIZER_JSON:
        cands.append(os.path.dirname(TOKENIZER_JSON) or ".")
    for root in (INPUT_ROOT, os.getcwd(), WORK_DIR):
        if os.path.isdir(root):
            for d, _, files in os.walk(root):
                if "tokenizer.json" in files:
                    cands.append(d)
    for d in cands:
        path = TOKENIZER_JSON or os.path.join(d, "tokenizer.json")
        try:
            tok = LegalByteBPE(path, os.path.join(d, "meta.json"))
        except Exception as e:
            log(f"skip tokenizer {path}: {e}")
            continue
        if tok.vocab_size != EXPECTED_VOCAB:
            log(f"skip tokenizer {path}: vocab {tok.vocab_size} != {EXPECTED_VOCAB} (probably another model's BPE)")
            continue
        log(f"tokenizer: {path} | vocab={tok.vocab_size} pad={tok.pad_id} unk={tok.unk_id} eos={tok.eos_id}")
        return tok
    raise FileNotFoundError("no 32k tokenizer.json found - attach the bpe-final.ipynb output dataset")


def tokenize_documents(docs, tok):
    """per document: list of token-id lists, one per sentence, each cut to MAX_SENT_TOK (never empty)"""
    out = []
    for d in docs:
        sents = []
        for s in d["text"]:
            ids = tok.encode(SENT_PREFIX + s.strip())[:MAX_SENT_TOK]
            sents.append(ids if ids else [tok.unk_id])
        d["sent_ids"] = sents
        out.append(sents)
    return out


def make_windows(sent_ids, rng=None):
    """
    Cover EVERY sentence exactly once as a *target*.  Each window = [left-context sentences][target block],
    total tokens <= MAX_LEN.  Returns a list of (ctx_start, tgt_start, tgt_end) sentence ranges
    (ctx_start <= tgt_start < tgt_end, half-open).
    rng=None -> deterministic (inference / dev);  rng given -> random target-block size (training augmentation).
    """
    n = len(sent_ids)
    lens = [len(s) for s in sent_ids]
    wins, i = [], 0
    while i < n:
        budget = TARGET_TOK if rng is None else rng.randint(*TRAIN_TARGET_RANGE)
        budget = min(budget, MAX_LEN)
        j, used = i, 0
        while j < n and (j == i or used + lens[j] <= budget):
            used += lens[j]
            j += 1
        # left context: as many previous sentences as still fit
        c, room = i, MAX_LEN - used
        while c > 0 and lens[c - 1] <= room:
            c -= 1
            room -= lens[c]
        wins.append((c, i, j))
        i = j
    return wins


def window_example(doc, label_ids, win, domain_id):
    c, a, b = win
    ids, ends, starts = [], [], []
    for k in range(c, b):
        starts.append(len(ids))
        ids.extend(doc["sent_ids"][k])
        ends.append(len(ids) - 1)
    n = len(doc["sent_ids"])
    return {
        "ids": ids,
        "ends": ends[a - c:],
        "starts": starts[a - c:],
        "labels": [label_ids[k] for k in range(a, b)] if label_ids is not None else [-100] * (b - a),
        "pos": [min(int(k / n * N_POS_BUCKETS), N_POS_BUCKETS - 1) for k in range(a, b)],
        "domain": domain_id,
        "sent_idx": list(range(a, b)),
    }


def collate(examples, pad_id, device):
    B = len(examples)
    T = max(len(e["ids"]) for e in examples)
    S = max(len(e["ends"]) for e in examples)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    ends = torch.zeros(B, S, dtype=torch.long)
    starts = torch.zeros(B, S, dtype=torch.long)
    labels = torch.full((B, S), -100, dtype=torch.long)
    pos = torch.zeros(B, S, dtype=torch.long)
    smask = torch.zeros(B, S, dtype=torch.bool)
    dom = torch.zeros(B, dtype=torch.long)
    for r, e in enumerate(examples):
        ids[r, :len(e["ids"])] = torch.tensor(e["ids"])           # right padding is safe: attention is causal
        m = len(e["ends"])
        ends[r, :m] = torch.tensor(e["ends"])
        starts[r, :m] = torch.tensor(e["starts"])
        labels[r, :m] = torch.tensor(e["labels"])
        pos[r, :m] = torch.tensor(e["pos"])
        smask[r, :m] = True
        dom[r] = e["domain"]
    return {k: v.to(device) for k, v in dict(ids=ids, ends=ends, starts=starts, labels=labels,
                                             pos=pos, smask=smask, dom=dom).items()}


def build_examples(docs, l2i, domain2id, rng=None):
    out = []
    for d in docs:
        label_ids = [l2i[x] for x in d["labels"]] if d.get("labels") is not None else None
        for w in make_windows(d["sent_ids"], rng):
            out.append(window_example(d, label_ids, w, domain2id[d["domain"]]))
    return out


# ============================================================================
# 5. DATA REPORT  (everything later steps need, in one json)
# ============================================================================
def compute_class_weights(counts):
    """counts: (n_domains, K) train label counts -> (n_domains, K) capped inverse-sqrt weights.
    Labels with ZERO examples in a domain get weight 0 there (they can never be a target in that domain)."""
    D, K = counts.shape
    pooled = counts.sum(0)
    W = np.zeros((D, K), dtype=np.float32)
    for d in range(D):
        c = counts[d] if PER_DOMAIN_WEIGHTS else pooled
        present = c > 0
        N, Kd = c.sum(), present.sum()
        W[d, present] = np.clip((N / (Kd * c[present])) ** CLASS_WEIGHT_POWER, *CLASS_WEIGHT_CLIP)
    return W


def rare_mask_from_counts(counts):
    """(D, K) bool: label is present in the domain's train set but below RARE_FRAC of its sentences"""
    share = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    return (counts > 0) & (share < RARE_FRAC)


def data_report(data, labels, l2i, tok):
    rep = {"labels": labels, "num_labels": len(labels), "splits": {}}
    for name, docs in data.items():
        tokenize_documents(docs, tok)
        sent_tok = np.array([len(s) for d in docs for s in d["sent_ids"]])
        raw_tok = [len(tok.encode(SENT_PREFIX + s.strip())) for d in docs for s in d["text"]]
        n_sent = [len(d["text"]) for d in docs]
        entry = {
            "documents": len(docs), "sentences": int(sum(n_sent)),
            "sentences_per_doc": {"mean": float(np.mean(n_sent)), "median": float(np.median(n_sent)),
                                  "max": int(max(n_sent))},
            "tokens_per_sentence": {"mean": float(np.mean(raw_tok)), "p95": float(np.percentile(raw_tok, 95)),
                                    "p99": float(np.percentile(raw_tok, 99)), "max": int(max(raw_tok))},
            f"sentences_truncated_at_{MAX_SENT_TOK}_tokens": int((np.array(raw_tok) > MAX_SENT_TOK).sum()),
            "tokens_per_doc_mean": float(sent_tok.sum() / len(docs)),
            "windows_per_epoch_inference_style": int(sum(len(make_windows(d["sent_ids"])) for d in docs)),
        }
        if name.endswith("_train") or name.endswith("_dev"):
            cnt = Counter(x for d in docs for x in d["labels"])
            entry["label_counts"] = {l: cnt.get(l, 0) for l in labels}
        rep["splits"][name] = entry

    # per-domain label counts from the TRAIN splits (one shared model, but weights/priors are domain specific)
    counts = np.zeros((len(DOMAINS), len(labels)), dtype=np.int64)
    for di, dom in enumerate(DOMAINS):
        c = Counter(x for d in data[f"{dom}_train"] for x in d["labels"])
        counts[di] = [c.get(l, 0) for l in labels]
    W = compute_class_weights(counts)
    rep["train_label_counts"] = {l: int(counts[:, i].sum()) for i, l in enumerate(labels)}
    rep["train_label_counts_by_domain"] = {dom: {l: int(counts[di, i]) for i, l in enumerate(labels)}
                                           for di, dom in enumerate(DOMAINS)}
    rep["class_weights_by_domain"] = {dom: {l: float(W[di, i]) for i, l in enumerate(labels)}
                                      for di, dom in enumerate(DOMAINS)}
    rep["labels_absent_from_train_by_domain"] = {dom: [l for i, l in enumerate(labels) if counts[di, i] == 0]
                                                 for di, dom in enumerate(DOMAINS)}
    rep["lora_window_note"] = (f"MAX_LEN={MAX_LEN}, TARGET_TOK={TARGET_TOK}, MAX_SENT_TOK={MAX_SENT_TOK}")

    log("=" * 90)
    log("DATA REPORT")
    log("=" * 90)
    for name, e in rep["splits"].items():
        log(f"{name:9s} docs={e['documents']:3d} sents={e['sentences']:6d} "
            f"tok/sent mean={e['tokens_per_sentence']['mean']:.1f} p99={e['tokens_per_sentence']['p99']:.0f} "
            f"max={e['tokens_per_sentence']['max']} | truncated={e[f'sentences_truncated_at_{MAX_SENT_TOK}_tokens']} "
            f"| windows/epoch={e['windows_per_epoch_inference_style']}")
    rare = rare_mask_from_counts(counts)
    log("\ntrain label counts and class weights per domain  (R = rare -> windows oversampled):")
    log(f"  {'':24s} " + " | ".join(f"{dom:>6s} count  share  weight" for dom in DOMAINS))
    for i, l in enumerate(labels):
        cells = []
        for di in range(len(DOMAINS)):
            share = 100 * counts[di, i] / max(counts[di].sum(), 1)
            cells.append(f"{counts[di, i]:12d} {share:5.2f}% {W[di, i]:6.2f}{'R' if rare[di, i] else ' '}")
        log(f"  {l:24s} " + " | ".join(cells))
    for di, dom in enumerate(DOMAINS):
        if rep["labels_absent_from_train_by_domain"][dom]:
            log(f"labels with ZERO {dom} train examples:", rep["labels_absent_from_train_by_domain"][dom])
        dev_c = Counter(x for d in data[f"{dom}_dev"] for x in d["labels"])
        missed = {l: n for l, n in dev_c.items() if counts[di, l2i[l]] == 0}
        if missed:
            log(f"WARNING: {dom}_dev contains labels unseen in {dom}_train (cannot be learned in-domain): {missed}")
    return rep, torch.tensor(W, dtype=torch.float32), counts


# ============================================================================
# 6. RR MODEL = Model 2 (+LoRA)  +  sentence head
# ============================================================================
class RRModel(nn.Module):
    def __init__(self, lm: DecoderLM, n_labels, n_domains=2):
        super().__init__()
        self.lm = lm
        d = lm.cfg.d_model
        d_in = d * (2 if POOLING == "last_mean" else 1)
        self.pos_emb = nn.Embedding(N_POS_BUCKETS, 16) if USE_POS_FEATURE else None
        self.dom_emb = nn.Embedding(n_domains, 8) if USE_DOMAIN_FEATURE else None
        d_in += (16 if USE_POS_FEATURE else 0) + (8 if USE_DOMAIN_FEATURE else 0)
        self.head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(d_in, HEAD_HIDDEN), nn.GELU(),
                                  nn.Dropout(HEAD_DROPOUT), nn.Linear(HEAD_HIDDEN, n_labels))

    def head_parameters(self):
        mods = [self.head] + [m for m in (self.pos_emb, self.dom_emb) if m is not None]
        return [p for m in mods for p in m.parameters()]

    def forward(self, b):
        h = self.lm.hidden(b["ids"])                                         # (B, T, d)
        B, S = b["ends"].shape
        d = h.size(-1)
        last = h.gather(1, b["ends"][:, :, None].expand(B, S, d))             # (B, S, d)
        feats = [last]
        if POOLING == "last_mean":
            csum = torch.cat([torch.zeros(B, 1, d, device=h.device, dtype=h.dtype), h.cumsum(1)], dim=1)
            hi = csum.gather(1, (b["ends"] + 1)[:, :, None].expand(B, S, d))
            lo = csum.gather(1, b["starts"][:, :, None].expand(B, S, d))
            cnt = (b["ends"] + 1 - b["starts"]).clamp(min=1)[:, :, None].to(h.dtype)
            feats.append((hi - lo) / cnt)
        if self.pos_emb is not None:
            feats.append(self.pos_emb(b["pos"]).to(h.dtype))
        if self.dom_emb is not None:
            feats.append(self.dom_emb(b["dom"])[:, None, :].expand(B, S, -1).to(h.dtype))
        return self.head(torch.cat(feats, dim=-1))                            # (B, S, n_labels)


# ---- adapter-only checkpoints (a few MB instead of the full 100M+ model) ----
def trainable_state(model):
    keep = {n for n, p in model.named_parameters() if p.requires_grad}
    state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if n in keep}
    state.update({n: b.detach().cpu().clone() for n, b in model.named_buffers() if n.endswith("rank_mask")})   # AdaLoRA: frozen budget
    return state


def save_adapter(model, path, extra):
    torch.save({"state": trainable_state(model), "lora": peft_meta(),
                "head": dict(pooling=POOLING, pos=USE_POS_FEATURE, dom=USE_DOMAIN_FEATURE), **extra}, path + ".tmp")
    os.replace(path + ".tmp", path)


def load_adapter(model, path):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    missing = model.load_state_dict(ck["state"], strict=False)
    assert not missing.unexpected_keys, missing.unexpected_keys
    return ck


# ---- find + load the Model 2 pretrained weights ----
def walk_input():
    if not os.path.isdir(INPUT_ROOT):
        return []
    return list(os.walk(INPUT_ROOT))


def find_model2_ckpt():
    if MODEL2_CKPT_PATH:
        return MODEL2_CKPT_PATH
    best = None                                                    # (val_loss, path): lowest-val-loss best.pt of this tag
    for d, _, files in walk_input():
        if "best.pt" in files and "best_meta.json" in files:
            try:
                meta = read_json(os.path.join(d, "best_meta.json"))
            except Exception:
                continue
            if meta.get("tag") == MODEL2_TAG and (best is None or meta["val_loss"] < best[0]):
                best = (meta["val_loss"], os.path.join(d, "best.pt"))
    if best is not None:
        return best[1]
    newest = None                                                  # (global_step, path): newest latest.pt of this tag
    for d, _, files in walk_input():
        if "latest.pt" in files and "latest_meta.json" in files:
            try:
                meta = read_json(os.path.join(d, "latest_meta.json"))
            except Exception:
                continue
            if meta.get("tag") == MODEL2_TAG and (newest is None or meta["global_step"] > newest[0]):
                newest = (meta["global_step"], os.path.join(d, "latest.pt"))
    if newest is not None:
        return newest[1]
    for d, _, files in walk_input():                               # last resort: a bare best.pt without its meta file
        if "best.pt" in files:
            log(f"WARNING: using {d}/best.pt without a best_meta.json - the tag inside the file is checked after loading")
            return os.path.join(d, "best.pt")
    raise FileNotFoundError("Model 2 checkpoint not found - attach the pretraining output (best.pt + best_meta.json)")


def build_base_model(tok, device):
    path = find_model2_ckpt()
    ck = torch.load(path, map_location="cpu", weights_only=False)
    if ck.get("tag") not in (None, MODEL2_TAG):
        raise ValueError(f"{path} belongs to another model (tag={ck.get('tag')!r}, expected {MODEL2_TAG!r})")
    cfg_dict = dict(ck["config"])
    cfg_dict["dropout"] = FT_DROPOUT                  # dropout has no parameters, so the weights still load
    cfg = ModelConfig(**cfg_dict)
    assert cfg.vocab_size == tok.vocab_size, (cfg.vocab_size, tok.vocab_size)
    lm = DecoderLM(cfg)
    lm.load_state_dict(ck["model"], strict=True)
    log(f"loaded Model 2 from {path} | {sum(p.numel() for p in lm.parameters()) / 1e6:.1f}M params | cfg={cfg}")
    del ck
    return lm.to(device)


# ============================================================================
# 7. TRAIN / EVAL / PREDICT
# ============================================================================
def macro_prf(y_true, y_pred, labels):
    """macro P/R/F1 over labels that occur in gold OR prediction (sklearn default); also per-class table"""
    from sklearn.metrics import precision_recall_fscore_support
    present = sorted(set(y_true) | set(y_pred))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=present, average=None, zero_division=0)
    per = {labels[c]: dict(p=float(p[i]), r=float(r[i]), f1=float(f[i]), support=int(s[i])) for i, c in enumerate(present)}
    return dict(macro_p=float(np.mean(p)), macro_r=float(np.mean(r)), macro_f1=float(np.mean(f)), per_class=per)


@torch.no_grad()
def predict_docs(model, docs, domain2id, pad_id, device, eval_bs=8, return_logits=False):
    """-> {doc_id: [label_id per sentence]} (and optionally {doc_id: (n, K) logits})"""
    model.eval()
    preds, logit_store = {}, {}
    for d in docs:
        n = len(d["sent_ids"])
        exs = [window_example(d, None, w, domain2id[d["domain"]]) for w in make_windows(d["sent_ids"])]
        out = np.full(n, -1, dtype=np.int64)
        lg = None
        for i in range(0, len(exs), eval_bs):
            chunk = exs[i:i + eval_bs]
            b = collate(chunk, pad_id, device)
            with amp_ctx():
                logits = model(b).float()
            if lg is None:
                lg = np.zeros((n, logits.size(-1)), dtype=np.float32)
            for r, e in enumerate(chunk):
                m = len(e["sent_idx"])
                lr = logits[r, :m].cpu().numpy()
                lg[e["sent_idx"]] = lr
                out[e["sent_idx"]] = lr.argmax(-1)
        assert (out >= 0).all() and len(out) == n, f"{d['id']}: not every sentence was predicted"
        preds[d["id"]] = out.tolist()
        logit_store[d["id"]] = lg
    return (preds, logit_store) if return_logits else preds


def evaluate_dev(model, dev_docs, labels, l2i, domain2id, pad_id, device):
    preds = predict_docs(model, dev_docs, domain2id, pad_id, device)
    res = {}
    for name, sel in (("ALL", dev_docs), ("CL", [d for d in dev_docs if d["domain"] == "CL"]),
                      ("IT", [d for d in dev_docs if d["domain"] == "IT"])):
        if not sel:
            continue
        yt = [l2i[x] for d in sel for x in d["labels"]]
        yp = [p for d in sel for p in preds[d["id"]]]
        res[name] = macro_prf(yt, yp, labels)
        res[name]["acc"] = float(np.mean(np.array(yt) == np.array(yp)))
    return res


def weighted_ce(logits, b, class_w):
    """class_w: (n_domains, K).  Each target token is weighted by ITS domain's weight for ITS label;
    normalised by the summed weight of the micro-batch (a weighted mean, so the loss scale stays ~constant)."""
    K = logits.size(-1)
    y = b["labels"]
    valid = y >= 0
    ce = F.cross_entropy(logits.float().reshape(-1, K), y.reshape(-1), ignore_index=-100,
                         reduction="none", label_smoothing=LABEL_SMOOTHING).view_as(y)
    w = class_w[b["dom"][:, None].expand_as(y), y.clamp(min=0)] * valid
    return (ce * w).sum() / w.sum().clamp(min=1e-6)


def oversample_rare(exs, rare_mask):
    out = []
    for e in exs:
        out.append(e)
        if any(rare_mask[e["domain"], y] for y in e["labels"] if y >= 0):
            out.extend([e] * (RARE_REPEAT - 1))
    return out


def make_epoch_examples(train_docs, l2i, domain2id, rng, rare_mask):
    exs = build_examples(train_docs, l2i, domain2id, rng)      # fresh random window boundaries
    return oversample_rare(exs, rare_mask) if OVERSAMPLE_RARE else exs


def build_post(counts):
    """per-domain log-priors (for logit adjustment) and the set of labels seen in each domain's train split"""
    allowed = counts > 0
    prior = counts / np.maximum(counts.sum(1, keepdims=True), 1)
    logp = np.log(np.where(allowed, prior, 1.0))
    for d in range(len(counts)):                    # unseen label: no boost (treated like the most frequent label)
        logp[d, ~allowed[d]] = logp[d, allowed[d]].max()
    return {"log_prior": logp.astype(np.float32), "allowed": allowed, "tau": 0.0, "use_mask": False}


def decode_logits(logit_store, docs, domain2id, post, tau=None, use_mask=None):
    """argmax of  logits - tau * log(prior_domain)   (tau=0: plain argmax), optionally forbidding labels unseen in-domain"""
    tau = post["tau"] if tau is None else tau
    use_mask = post["use_mask"] if use_mask is None else use_mask
    preds = {}
    for d in docs:
        di = domain2id[d["domain"]]
        lg = logit_store[d["id"]].copy()
        if tau:
            lg -= tau * post["log_prior"][di][None, :]
        if use_mask:
            lg[:, ~post["allowed"][di]] = -1e9
        preds[d["id"]] = lg.argmax(-1).tolist()
    return preds


def calibrate_postproc(model, dev_docs, labels, l2i, domain2id, pad_id, device, counts):
    """Two cheap knobs tuned on dev macro-F1: tau (prior correction strength) and the unseen-label mask.
    Only 2 degrees of freedom, so it cannot overfit the small dev set the way per-class biases would."""
    _, store = predict_docs(model, dev_docs, domain2id, pad_id, device, return_logits=True)
    post = build_post(counts)
    yt = [l2i[x] for d in dev_docs for x in d["labels"]]
    best = None
    for use_mask in (False, True):
        for tau in TAU_GRID:
            pr = decode_logits(store, dev_docs, domain2id, post, tau, use_mask)
            yp = [p for d in dev_docs for p in pr[d["id"]]]
            f1 = macro_prf(yt, yp, labels)["macro_f1"]
            log(f"  calibration: tau={tau:.2f} mask_unseen={use_mask!s:5s} -> dev macro-F1 {f1:.4f}")
            if best is None or f1 > best[0] + 1e-4:          # ties go to the simpler setting (tried first)
                best = (f1, tau, use_mask)
    post["tau"], post["use_mask"] = best[1], best[2]
    log(f"calibration chosen: tau={best[1]} mask_unseen={best[2]} (dev macro-F1 {best[0]:.4f})")
    return post, best[0]


def lr_scale(step, total, warmup):
    if step < warmup:
        return (step + 1) / max(1, warmup)
    prog = (step - warmup) / max(1, total - warmup)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))


def train(model, train_docs, dev_docs, labels, l2i, domain2id, class_w, tok, device, rare_mask):
    os.makedirs(CKPT_DIR, exist_ok=True)
    K = len(labels)
    lora_params = [p for n, p in model.lm.named_parameters() if p.requires_grad]
    head_params = model.head_parameters()
    n_lora, n_head = sum(p.numel() for p in lora_params), sum(p.numel() for p in head_params)
    n_all = sum(p.numel() for p in model.lm.parameters())
    log(f"trainable: {PEFT_METHOD}(+norm) {n_lora / 1e6:.2f}M + head {n_head / 1e6:.2f}M "
        f"= {100 * (n_lora + n_head) / (n_all + n_head):.2f}% of all parameters")

    lr_peft = {"adalora": LR_ADALORA, "rslora": LR_RSLORA}.get(PEFT_METHOD, LR_LORA)
    opt = torch.optim.AdamW(
        [{"params": lora_params, "lr": lr_peft, "base_lr": lr_peft, "weight_decay": 0.0},
         {"params": head_params, "lr": LR_HEAD, "base_lr": LR_HEAD, "weight_decay": WEIGHT_DECAY}],
        betas=(0.9, 0.999), eps=1e-8)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP["enabled"] and AMP["dtype"] == torch.float16)
    class_w = class_w.to(device)

    rng = random.Random(SEED)
    steps_per_epoch = max(1, math.ceil(len(make_epoch_examples(train_docs, l2i, domain2id, random.Random(SEED), rare_mask)) / (MICRO_BS * ACCUM)))
    total_steps, warmup = EPOCHS * steps_per_epoch, int(WARMUP_FRAC * EPOCHS * steps_per_epoch)
    log(f"epochs={EPOCHS} steps/epoch~{steps_per_epoch} total~{total_steps} "
        f"effective batch={MICRO_BS * ACCUM} windows")
    allocator = AdaLoRAAllocator(model.lm, total_steps) if PEFT_METHOD == "adalora" else None
    if allocator is not None:
        log(allocator.describe())

    best_f1, best_epoch, bad, gstep = -1.0, -1, 0, 0
    log_path = os.path.join(CKPT_DIR, "train_log.csv")
    with open(log_path, "w", newline="") as f:
        csv.writer(f).writerow(["epoch", "train_loss", "dev_macro_f1", "dev_cl_f1", "dev_it_f1", "dev_acc", "sec"])

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        exs = make_epoch_examples(train_docs, l2i, domain2id, rng, rare_mask)   # fresh windows + rare-label oversampling
        rng.shuffle(exs)
        run, nrun, run_reg = 0.0, 0, 0.0
        epoch_gstep0 = gstep
        for i in range(0, len(exs), MICRO_BS * ACCUM):
            group = exs[i:i + MICRO_BS * ACCUM]
            n_micro = math.ceil(len(group) / MICRO_BS)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_scale(gstep, total_steps, warmup)
            for j in range(0, len(group), MICRO_BS):
                b = collate(group[j:j + MICRO_BS], tok.pad_id, device)
                with amp_ctx():
                    logits = model(b)
                loss = weighted_ce(logits, b, class_w)      # per-domain capped inverse-sqrt weights
                scaler.scale(loss / n_micro).backward()
                run += loss.item()
                nrun += 1
            if allocator is not None:                       # AdaLoRA: orthogonality penalty on P (lora_B) and Q (lora_A), once per step
                reg = allocator.orth_reg()
                scaler.scale(ADALORA_ORTH_REG * reg).backward()
                run_reg += reg.item()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(lora_params + head_params, GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            if allocator is not None and torch.isfinite(gn):   # sensitivity scores + budget pruning (needs the grads -> before zero_grad)
                allocator.update_and_allocate(gstep)
            opt.zero_grad(set_to_none=True)
            gstep += 1

        res = evaluate_dev(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device)
        f1 = res["ALL"]["macro_f1"]
        row = [epoch + 1, f"{run / max(nrun, 1):.4f}", f"{f1:.4f}", f"{res.get('CL', {}).get('macro_f1', float('nan')):.4f}",
               f"{res.get('IT', {}).get('macro_f1', float('nan')):.4f}", f"{res['ALL']['acc']:.4f}", f"{time.time() - t0:.0f}"]
        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow(row)
        improved = f1 > best_f1 and (allocator is None or allocator.frozen or not ADALORA_SELECT_AFTER_FREEZE)
        log(f"[epoch {epoch + 1:2d}/{EPOCHS}] loss {row[1]} | dev macro-F1 {row[2]} (CL {row[3]} / IT {row[4]}) "
            f"acc {row[5]} | {row[6]}s {'<-- best' if improved else ''}")
        if allocator is not None:
            log(f"   {allocator.status(gstep)} | orth-reg {run_reg / max(1, gstep - epoch_gstep0):.3f}")
        if improved:
            best_f1, best_epoch, bad = f1, epoch + 1, 0
            save_adapter(model, os.path.join(CKPT_DIR, "best_adapter.pt"),
                         {"epoch": epoch + 1, "dev_macro_f1": f1, "budget_frozen": bool(allocator is not None and allocator.frozen)})
        else:
            bad += 1
            if allocator is not None and not allocator.frozen:
                bad = 0                      # AdaLoRA: the budget is still shrinking -> dev F1 is not comparable yet, no early stop
            if bad >= PATIENCE:
                log(f"early stop: no dev improvement for {PATIENCE} epochs")
                break
    if best_epoch < 0:                       # (AdaLoRA) the budget never froze -> keep the last state so the run still yields an adapter
        save_adapter(model, os.path.join(CKPT_DIR, "best_adapter.pt"), {"epoch": EPOCHS, "dev_macro_f1": f1, "budget_frozen": False})
        best_f1, best_epoch = f1, EPOCHS
        log("WARNING: the AdaLoRA budget was never frozen (training ended too early) - saved the last state instead")
    log(f"best dev macro-F1 {best_f1:.4f} at epoch {best_epoch}")
    return best_f1


def write_submission(preds_by_id, data, labels):
    expected = {d["id"]: len(d["text"]) for k in ("CL_test", "IT_test") for d in data[k]}
    assert set(preds_by_id) == set(expected), "prediction ids differ from the test ids"
    out, valid = {}, set(labels)
    for doc_id, n in expected.items():                       # test-file order, CL then IT
        lab = [labels[i] for i in preds_by_id[doc_id]]
        assert len(lab) == n and all(l in valid for l in lab), doc_id
        out[doc_id] = lab
    os.makedirs(WORK_DIR, exist_ok=True)
    pred_path, zip_path = os.path.join(WORK_DIR, "prediction.json"), os.path.join(WORK_DIR, "submission.zip")
    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    dist = Counter(l for v in out.values() for l in v)
    log(f"wrote {pred_path} and {zip_path} | {len(out)} docs, {sum(len(v) for v in out.values())} sentences")
    log("predicted label distribution:", dict(dist.most_common()))


# ============================================================================
# 8. MAIN
# ============================================================================
def run_self_checks(model, device):
    """LoRA must be an exact no-op at init, merge/unmerge must be lossless, and only adapters+head may train"""
    model.eval()
    ids = torch.randint(3, model.lm.cfg.vocab_size, (1, 64), device=device)
    saved, AMP["enabled"] = AMP["enabled"], False
    try:
        with torch.no_grad():
            ref = model.lm.hidden(ids)
            for m in lora_modules(model):                         # make the adapter non-zero so merge is actually tested
                _perturb(m)
            adapted = model.lm.hidden(ids)
            merge_lora(model)
            merged = model.lm.hidden(ids)
            unmerge_lora(model)
            back = model.lm.hidden(ids)
            for m in lora_modules(model):
                _unperturb(m)
        assert (adapted - ref).abs().max() > 1e-6, "LoRA had no effect"
        assert (adapted - merged).abs().max() < 1e-3, "merged LoRA differs from adapted forward"
        assert (adapted - back).abs().max() < 1e-4, "unmerge is not lossless"
    finally:
        AMP["enabled"] = saved
    frozen_bad = [n for n, p in model.lm.named_parameters()
                  if p.requires_grad and "lora_" not in n and not n.startswith("final_norm")]
    assert not frozen_bad, f"base weights are trainable: {frozen_bad[:3]}"
    log("self-checks passed: LoRA is a no-op at init, merge/unmerge are lossless, base weights are frozen")


def run_rslora_checks(model):
    """every adapted matrix must use scaling = alpha / sqrt(r) (and NOT alpha / r)"""
    mods = [m for m in model.lm.modules() if isinstance(m, LoRALinear)]
    assert mods, "no LoRALinear found - call inject_lora first"
    want = RSLORA_ALPHA / math.sqrt(RSLORA_R)
    for m in mods:
        assert m.rank_stabilized and m.r == RSLORA_R, "module is not an rsLoRA module of rank RSLORA_R"
        assert math.isclose(m.scaling, want, rel_tol=1e-9), f"scaling {m.scaling} != alpha/sqrt(r) = {want}"
    log(f"rsLoRA self-checks passed: all {len(mods)} matrices use scaling = alpha/sqrt(r) = {RSLORA_ALPHA:g}/sqrt({RSLORA_R}) = {want:g} "
        f"(plain LoRA at the same r and alpha would use {RSLORA_ALPHA / RSLORA_R:g})")


def main():
    set_seed()
    os.makedirs(WORK_DIR, exist_ok=True)
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    if use_cuda:
        AMP["enabled"] = True
        AMP["dtype"] = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
        log(f"GPU: {torch.cuda.get_device_name()} | autocast dtype: {AMP['dtype']}")
    else:
        log("WARNING: no GPU visible - training will be very slow")

    data, labels, l2i = load_phase3()
    tok = load_tokenizer()
    domain2id = {d: i for i, d in enumerate(DOMAINS)}

    report, class_w, counts = data_report(data, labels, l2i, tok)
    rare_mask = rare_mask_from_counts(counts)
    with open(os.path.join(WORK_DIR, "data_report.json"), "w") as f:
        json.dump(report, f, indent=2)

    lm = build_base_model(tok, device)
    if PEFT_METHOD == "adalora":
        n_wrapped = inject_adalora(lm)
        log(f"AdaLoRA injected into {n_wrapped} linear layers (r_init={ADALORA_INIT_R} -> avg r={ADALORA_TARGET_R}, "
            f"alpha={ADALORA_ALPHA}, targets={LORA_TARGETS})")
    elif PEFT_METHOD == "rslora":
        n_wrapped = inject_lora(lm, r=RSLORA_R, alpha=RSLORA_ALPHA, rank_stabilized=True)
        log(f"rsLoRA injected into {n_wrapped} linear layers (r={RSLORA_R}, alpha={RSLORA_ALPHA:g}, "
            f"scaling=alpha/sqrt(r)={RSLORA_ALPHA / math.sqrt(RSLORA_R):g}, targets={LORA_TARGETS})")
    else:
        n_wrapped = inject_lora(lm)
        log(f"LoRA injected into {n_wrapped} linear layers (r={LORA_R}, alpha={LORA_ALPHA}, targets={LORA_TARGETS})")
    model = RRModel(lm, len(labels)).to(device)
    run_self_checks(model, device)
    if PEFT_METHOD == "rslora":
        run_rslora_checks(model)
    if PEFT_METHOD == "adalora":
        run_adalora_checks(model, device)

    train_docs = data["CL_train"] + data["IT_train"]
    dev_docs = data["CL_dev"] + data["IT_dev"]
    train(model, train_docs, dev_docs, labels, l2i, domain2id, class_w, tok, device, rare_mask)

    # final dev report with the best adapter
    ck = load_adapter(model, os.path.join(CKPT_DIR, "best_adapter.pt"))
    res = evaluate_dev(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device)
    log(f"\nBEST adapter (epoch {ck['epoch']}) on dev: macro-P {res['ALL']['macro_p']:.4f} "
        f"macro-R {res['ALL']['macro_r']:.4f} macro-F1 {res['ALL']['macro_f1']:.4f}")
    for l, v in res["ALL"]["per_class"].items():
        log(f"  {l:24s} P {v['p']:.3f} R {v['r']:.3f} F1 {v['f1']:.3f} support {v['support']}")
    with open(os.path.join(WORK_DIR, "dev_report.json"), "w") as f:
        json.dump(res, f, indent=2)
    if PEFT_METHOD == "adalora":
        report_rank_allocation(model, os.path.join(WORK_DIR, "rank_allocation.json"))

    post = build_post(counts)
    if POSTHOC_CALIBRATION:
        log("\npost-hoc calibration on dev:")
        post, cal_f1 = calibrate_postproc(model, dev_docs, labels, l2i, domain2id, tok.pad_id, device, counts)
        _, dstore = predict_docs(model, dev_docs, domain2id, tok.pad_id, device, return_logits=True)
        dpred = decode_logits(dstore, dev_docs, domain2id, post)
        for name in ("ALL",) + tuple(DOMAINS):
            sel = dev_docs if name == "ALL" else [d for d in dev_docs if d["domain"] == name]
            r = macro_prf([l2i[x] for d in sel for x in d["labels"]], [p for d in sel for p in dpred[d["id"]]], labels)
            log(f"  calibrated dev {name:3s}: macro-P {r['macro_p']:.4f} macro-R {r['macro_r']:.4f} macro-F1 {r['macro_f1']:.4f}")
        with open(os.path.join(WORK_DIR, "postproc.json"), "w") as f:
            json.dump({"tau": post["tau"], "mask_unseen": bool(post["use_mask"]), "dev_macro_f1": cal_f1}, f, indent=2)

    if RUN_PREDICTION:
        test_docs = data["CL_test"] + data["IT_test"]
        tokenize_documents(test_docs, tok)
        merge_lora(model)                         # fold the adapters into the weights: no extra inference cost
        _, tstore = predict_docs(model, test_docs, domain2id, tok.pad_id, device, return_logits=True)
        preds = decode_logits(tstore, test_docs, domain2id, post)
        write_submission(preds, data, labels)
    log("done.")


if __name__ == "__main__":
    main()